In [ ]:
# ==============================================================================
# NOTEBOOK: 00_batch_pipeline_and_audit
# DESCRIPCIÓN: Pipeline Batch (Ingesta + RAW + Bronze) + Inventario + Auditoría
# ==============================================================================
from pyspark.sql import functions as F

print("================================================================================")
print("🚀 INICIANDO CANALIZACIÓN DE DATOS BATCH, INVENTARIO Y AUDITORÍA DE DATOS")
print("================================================================================\n")

# ------------------------------------------------------------------------------
# 1. FASE DE INGESTA LANDING BATCH
# ------------------------------------------------------------------------------
print("--- 1. FASE DE INGESTA LANDING BATCH ---")
ingesters = [
    ("NASA FIRMS Historical (7 días)", "01_landing_nasa_historical"),  # Corregida la errata (nasa)
    ("OpenStreetMap (Red Viaria)", "01_landing_osm_data")
]

for name, nb in ingesters:
    try:
        # Aumentamos el timeout a 300s para evitar el corte en la llamada a Overpass API
        res = mssparkutils.notebook.run(nb, timeout_seconds=300)
        print(f"✅ Ingesta {name} completada. Salida: {res}")
    except Exception as e:
        print(f"❌ Error en ingesta {name}: {e}")

# 2. FASE PROCESAMIENTO RAW BATCH
print("\n--- 2. FASE PROCESAMIENTO RAW BATCH (JSON/CSV -> Parquet) ---")
try:
    mssparkutils.notebook.run("02_raw_batch")
    print("✅ Archivos RAW Batch actualizados en Parquet.")
except Exception as e:
    print(f"❌ Error en la capa RAW Batch: {e}")

# 3. FASE PROMOCIÓN A BRONZE BATCH
print("\n--- 3. FASE PROMOCIÓN A BRONZE BATCH (Parquet -> Tablas Delta) ---")
try:
    mssparkutils.notebook.run("03_bronze_batch")
    print("✅ Tablas Delta Bronze Batch actualizadas.")
except Exception as e:
    print(f"❌ Error en la capa Bronze Batch: {e}")

# 4. INVENTARIO DE ARCHIVOS BATCH
print("\n" + "="*80)
print("📦 INVENTARIO Y CONTEO DETALLADO DE ARCHIVOS BATCH")
print("="*80)

def list_and_count_files(path: str):
    file_list = []
    try:
        items = mssparkutils.fs.ls(path)
        for item in items:
            if item.isDir:
                sub_count, sub_list = list_and_count_files(item.path)
                file_list.extend(sub_list)
            elif not item.name.startswith("_") and not item.name.startswith("."):
                file_list.append(item.path)
    except Exception:
        pass
    return len(file_list), file_list

landing_batch_dirs = [
    ("NASA Histórico (CSV)", "Files/landing/batch/nasa_historical"),
    ("OpenStreetMap (JSON)", "Files/landing/batch/osm_roads")
]

print("\n📂 LANDING BATCH:")
for name, path in landing_batch_dirs:
    total_f, files = list_and_count_files(path)
    print(f"\n   • {name} [{path}]: Total = {total_f:,} archivos.")
    if files:
        for f_path in files[:5]:
            print(f"       - {f_path}")
        if total_f > 5:
            print(f"       ... y {total_f - 5:,} archivos más.")

raw_batch_dirs = [
    ("NASA Histórico RAW", "Files/raw/batch/nasa_historical"),
    ("OpenStreetMap RAW", "Files/raw/batch/osm_roads")
]

print("\n📂 RAW BATCH:")
for name, path in raw_batch_dirs:
    total_p, files = list_and_count_files(path)
    print(f"\n   • {name} [{path}]: Total = {total_p:,} archivos Parquet.")
    if files:
        for f_path in files[:5]:
            print(f"       - {f_path}")
        if total_p > 5:
            print(f"       ... y {total_p - 5:,} archivos más.")

print("\n🛢️ BRONZE BATCH:")
for t_name in ["bronze_nasa_historical", "bronze_osm_roads"]:
    try:
        cnt = spark.table(t_name).count()
        print(f"   • {t_name:<25}: {cnt:,} registros Delta.")
    except Exception:
        print(f"   • {t_name:<25}: ❌ No encontrada en catálogo.")

# 5. AUDITORÍA BATCH
print("\n" + "="*80)
print("📊 AUDITORÍA DE DATOS Y MARCAS DE TRAZABILIDAD EN BRONZE BATCH")
print("="*80 + "\n")

def run_bronze_batch_audit():
    batch_tables = ["bronze_nasa_historical", "bronze_osm_roads"]
    meta_cols = ["landing_source_file", "ingestion_timestamp", "updated_source_file", "updated_timestamp"]

    for table_name in batch_tables:
        print("-" * 80)
        print(f"🔎 AUDITANDO TABLA BATCH: {table_name.upper()}")
        print("-" * 80)
        try:
            df = spark.table(table_name)
            total_records = df.count()
            print(f"🔹 Nº Total de Registros: {total_records:,}")

            if total_records == 0:
                continue

            print("\n🛡️ AUDITORÍA DE METADATOS Y MARCAS TEMPORALES:")
            if "landing_source_file" in df.columns:
                dist_f = df.select(F.countDistinct("landing_source_file")).collect()[0][0]
                print(f"   • landing_source_file  [🟢 INGESTA INICIAL]: {dist_f:,} ficheros origen distintos.")

            if "ingestion_timestamp" in df.columns:
                ing_stats = df.select(F.min("ingestion_timestamp").alias("min_ing"), F.max("ingestion_timestamp").alias("max_ing")).collect()[0]
                print(f"   • ingestion_timestamp  [🟢 INGESTA INICIAL]: Desde [{ing_stats['min_ing']}] Hasta [{ing_stats['max_ing']}]")

            if "updated_source_file" in df.columns:
                upd_cnt = df.filter(F.col("updated_source_file").isNotNull()).count()
                print(f"   • updated_source_file  [🔴 ACTUALIZACIÓN]: {upd_cnt:,} registros modificados.")

            print("\n🔑 Completitud por Columna (primeras 10):")
            for col_name, dtype in df.dtypes[:10]:
                n_cnt = df.filter(F.col(col_name).isNull()).count()
                completeness = round(((total_records - n_cnt) / total_records) * 100, 2)
                print(f"   • {col_name:<30} ({dtype:<10}): Completitud: {completeness:>6.2f}%")

            print("\n📄 Muestra de Registros:")
            sample_cols = [c for c in meta_cols if c in df.columns]
            other_cols = [c for c in df.columns if c not in meta_cols][:3]
            df.select(other_cols + sample_cols).show(3, truncate=40)
            print("\n")
        except Exception as e:
            print(f"❌ Error auditando la tabla {table_name}: {e}\n")

run_bronze_batch_audit()

In [ ]:
# ==============================================================================
# AUDITORÍA UNIFICADA DE CATALOGO BRONZE (5 FUENTES)
# ==============================================================================
import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, DateType, TimestampType, StringType

bronze_tables = {
    "bronze_dgt_traffic": ["record_id"],
    "bronze_weather": ["latitude", "longitude", "generationtime_ms"],
    "bronze_nasa_nrt": ["latitude", "longitude", "acq_date", "acq_time"],
    "bronze_nasa_historical": ["latitude", "longitude", "acq_date", "acq_time"],
    "bronze_osm_roads": ["id"]
}

print("==================================================================================")
print("🔍 INICIANDO AUDITORÍA DETALLADA Y TÉCNICA DEL CATÁLOGO BRONZE (5 FUENTES)")
print("==================================================================================\n")

for table_name, pk_cols in bronze_tables.items():
    print("=" * 90)
    print(f"🛢️ TABLA: {table_name.upper()}")
    print("=" * 90)
    
    try:
        df = spark.table(table_name)
    except Exception as e:
        print(f"❌ La tabla '{table_name}' no existe en el catálogo.\n")
        continue

    total_count = df.count()
    print(f"📊 Volumetría Total: {total_count:,} filas | Total Columnas: {len(df.columns)}")
    print(f"🔑 Clave Primaria Teórica: {pk_cols}")
    
    # 1. Validación de Clave Primaria y Duplicados
    valid_pks = [c for c in pk_cols if c in df.columns]
    if valid_pks:
        pk_distinct_count = df.select(valid_pks).distinct().count()
        dups_count = total_count - pk_distinct_count
        print(f"🎯 Registros Únicos por PK: {pk_distinct_count:,} | Duplicados por PK: {dups_count:,} ({round((dups_count/total_count)*100, 2) if total_count > 0 else 0}%)")
    else:
        print("⚠️ No se encontraron las columnas de PK teóricas en la tabla.")

    # 2. Detección de Campos Fecha/Hora
    date_candidates = []
    for col_name, dtype in df.dtypes:
        col_lower = col_name.lower()
        if any(k in col_lower for k in ["date", "time", "timestamp", "creation", "generation"]):
            date_candidates.append(f"{col_name} ({dtype})")
    print(f"📅 Campos Detectados como Fecha/Hora: {date_candidates if date_candidates else 'Ninguno'}")

    # 3. Análisis Columna por Columna
    print("\n📋 Detalle Extensivo por Columna:")
    print(f"{'Columna':<35} | {'Tipo':<12} | {'Nulos':<8} | {'% Nulos':<8} | {'Únicos':<10} | {'Rango / Comentario':<30}")
    print("-" * 115)

    for field in df.schema.fields:
        col_name = field.name
        dtype_str = field.dataType.simpleString()

        # Conteo de Nulos y Únicos
        null_count = df.filter(F.col(col_name).isNull() | F.nanvl(F.col(col_name), F.lit(None)).isNull() if isinstance(field.dataType, NumericType) else F.col(col_name).isNull()).count()
        null_pct = round((null_count / total_count) * 100, 1) if total_count > 0 else 0.0
        
        # Conteo de Valores Distintos
        distinct_count = df.select(col_name).distinct().count()

        # Rango Mínimo y Máximo para Numéricos/Fechas
        range_str = "-"
        if isinstance(field.dataType, (NumericType, DateType, TimestampType)):
            stats = df.select(F.min(col_name).alias("min_v"), F.max(col_name).alias("max_v")).collect()[0]
            min_v, max_v = stats["min_v"], stats["max_v"]
            if min_v is not None and max_v is not None:
                range_str = f"[{min_v} .. {max_v}]"
        elif isinstance(field.dataType, StringType) and distinct_count <= 5:
            # Mostrar muestra si son pocos valores categóricos
            sample_vals = [row[col_name] for row in df.select(col_name).distinct().limit(3).collect() if row[col_name] is not None]
            range_str = f"Ej: {sample_vals}"

        print(f"{col_name:<35} | {dtype_str:<12} | {null_count:<8,} | {null_pct:>6.1f}% | {distinct_count:<10,} | {range_str[:30]:<30}")

    print("\n")